## **0. Library import**

In [1]:
import os
import sys

# Add the root path into the python path
root_path = os.path.abspath(os.path.join(".."))
if not root_path in sys.path:
    sys.path.insert(0, root_path)

In [2]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

from src.config import BRFSS_FILTERING_FILE_PATH, TRAIN_SIZE, RANDOM_STATE, \
    SGD_ALPHA, SGD_MAX_ITER, SGD_LOSS, SGD_PENALTY, SGD_LEARNING_RATE, \
    LR_C, LR_MAX_ITER, LR_SOLVER, \
    RF_N_ESTIMATORS, RF_MAX_DEPTH, \
    NB_ALPHA, NB_TYPE, NB_BINARIZE, NB_VAR_SMOOTHING, \
    OVER_SAMPLING_STRATEGY, UNDER_SAMPLING_STRATEGY

from src.features import DiabetesFeatureEngineering

from src.balancing.over_sampler import OverSamplingBalancer
from src.balancing.under_sampler import UnderSamplingBalancer
from src.balancing.hyprid import HybridSamplingBalancer

from src.evaluate import plot_roc_auc, compare_models
from src.models.sgd_classifier import DiabetesSGDClassifier
from src.models.logistic_regression import DiabetesLogisticRegression
from src.models.random_forest import DiabetesRandomForest
from src.models.naive_bayes import DiabetesNaiveBayes

## **1. Load dataset**

In [3]:
df = pd.read_csv(BRFSS_FILTERING_FILE_PATH)
df.shape

(787646, 21)

## **2. Feature engineering**

In [4]:
diabetes_feature_engineering = DiabetesFeatureEngineering()
processed_df = diabetes_feature_engineering.process_all(df)

2025-07-20 13:31:38,284 - [src.features] - INFO - DiabetesFeatureEngineering initialized successfully
2025-07-20 13:31:38,286 - [src.features] - INFO - ============================================================
2025-07-20 13:31:38,287 - [src.features] - INFO - STARTING COMPLETE FEATURE ENGINEERING PIPELINE
2025-07-20 13:31:38,288 - [src.features] - INFO - ============================================================
2025-07-20 13:31:38,289 - [src.features] - INFO - Initial dataset shape: (787646, 21)
2025-07-20 13:31:38,290 - [src.features] - INFO - Starting null values removal process...
2025-07-20 13:31:38,310 - [src.features] - INFO - No null values found in the dataset
2025-07-20 13:31:38,378 - [src.features] - INFO - Null values removal completed. Removed 0 rows (0.00%)
2025-07-20 13:31:38,379 - [src.features] - INFO - Dataset shape: 787646 -> 787646 rows
2025-07-20 13:31:38,379 - [src.features] - INFO - After null removal: (787646, 21)
2025-07-20 13:31:38,379 - [src.features] - 

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 787646 entries, 0 to 787645
Data columns (total 21 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   Diabetes              787646 non-null  float64
 1   HighBP                787646 non-null  float64
 2   HighChol              787646 non-null  float64
 3   CholCheck             787646 non-null  float64
 4   BMI                   787646 non-null  float64
 5   Smoker                787646 non-null  float64
 6   Stroke                787646 non-null  float64
 7   HeartDiseaseorAttack  787646 non-null  float64
 8   PhysActivity          787646 non-null  float64
 9   HvyAlcoholConsump     787646 non-null  float64
 10  AnyHealthcare         787646 non-null  float64
 11  NoDocbcCost           787646 non-null  float64
 12  GenHlth               787646 non-null  float64
 13  MentHlth              787646 non-null  float64
 14  PhysHlth              787646 non-null  float64
 15  

In [6]:
df["Diabetes"].value_counts()

Diabetes
0.0    657902
2.0    112781
1.0     16963
Name: count, dtype: int64

## **3. Split train and test data**

In [7]:
X = df.drop("Diabetes", axis=1)
y = df["Diabetes"]

In [8]:
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=TRAIN_SIZE, random_state=RANDOM_STATE, stratify=y)
print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

X_train shape: (630116, 20)
X_test shape: (157530, 20)
y_train shape: (630116,)
y_test shape: (157530,)


## **4. Data balancing**

### **4.1 Over sampling**

In [9]:
over_sampling_balancer = OverSamplingBalancer()

2025-07-20 13:31:41,977 - [src.balancing.over_sampler] - INFO - OverSamplingBalancer initialized successfully


#### **4.1.1 Random Over Sampling**

In [10]:
ros_X_train, ros_y_train = over_sampling_balancer.apply_random_oversampling(
    X_train,
    y_train,
    OVER_SAMPLING_STRATEGY
)

2025-07-20 13:31:41,998 - [src.balancing.over_sampler] - INFO - Starting Random Over Sampling process...
2025-07-20 13:31:42,010 - [src.balancing.over_sampler] - INFO - Original class distribution:
2025-07-20 13:31:42,015 - [src.balancing.over_sampler] - INFO -   - Class 0.0: 526321 samples
2025-07-20 13:31:42,016 - [src.balancing.over_sampler] - INFO -   - Class 1.0: 13570 samples
2025-07-20 13:31:42,016 - [src.balancing.over_sampler] - INFO -   - Class 2.0: 90225 samples
2025-07-20 13:31:42,216 - [src.balancing.over_sampler] - INFO - New class distribution after Random Over Sampling:
2025-07-20 13:31:42,216 - [src.balancing.over_sampler] - INFO -   - Class 0.0: 526321 samples (+0)
2025-07-20 13:31:42,217 - [src.balancing.over_sampler] - INFO -   - Class 1.0: 50000 samples (+36430)
2025-07-20 13:31:42,217 - [src.balancing.over_sampler] - INFO -   - Class 2.0: 100000 samples (+9775)
2025-07-20 13:31:42,217 - [src.balancing.over_sampler] - INFO - Dataset size: 630116 -> 676321 samples (

In [11]:
ros_y_train.value_counts()

Diabetes
0.0    526321
2.0    100000
1.0     50000
Name: count, dtype: int64

#### **4.1.2 SMOTE**

In [12]:
smote_X_train, smote_y_train = over_sampling_balancer.apply_smote(
    X_train,
    y_train,
    OVER_SAMPLING_STRATEGY
)

2025-07-20 13:31:42,243 - [src.balancing.over_sampler] - INFO - Starting SMOTE process...
2025-07-20 13:31:42,248 - [src.balancing.over_sampler] - INFO - Original class distribution:
2025-07-20 13:31:42,249 - [src.balancing.over_sampler] - INFO -   - Class 0.0: 526321 samples
2025-07-20 13:31:42,250 - [src.balancing.over_sampler] - INFO -   - Class 1.0: 13570 samples
2025-07-20 13:31:42,250 - [src.balancing.over_sampler] - INFO -   - Class 2.0: 90225 samples
2025-07-20 13:31:55,790 - [src.balancing.over_sampler] - INFO - New class distribution after SMOTE:
2025-07-20 13:31:55,790 - [src.balancing.over_sampler] - INFO -   - Class 0.0: 526321 samples (+0 synthetic)
2025-07-20 13:31:55,791 - [src.balancing.over_sampler] - INFO -   - Class 1.0: 50000 samples (+36430 synthetic)
2025-07-20 13:31:55,792 - [src.balancing.over_sampler] - INFO -   - Class 2.0: 100000 samples (+9775 synthetic)
2025-07-20 13:31:55,792 - [src.balancing.over_sampler] - INFO - Dataset size: 630116 -> 676321 samples (

In [13]:
smote_y_train.value_counts()

Diabetes
0.0    526321
2.0    100000
1.0     50000
Name: count, dtype: int64

### **4.2 Under Sampling**

In [14]:
under_sampling_balancer = UnderSamplingBalancer()

2025-07-20 13:31:55,827 - [src.balancing.under_sampler] - INFO - UnderSamplingBalancer initialized successfully


#### **4.2.1 Random Under Sampling**

In [15]:
rus_X_train, rus_y_train = under_sampling_balancer.apply_random_undersampling(
    X_train,
    y_train,
    UNDER_SAMPLING_STRATEGY
)

2025-07-20 13:31:55,840 - [src.balancing.under_sampler] - INFO - Starting Random Under Sampling process...
2025-07-20 13:31:55,852 - [src.balancing.under_sampler] - INFO - Original class distribution:
2025-07-20 13:31:55,854 - [src.balancing.under_sampler] - INFO -   - Class 0.0: 526321 samples
2025-07-20 13:31:55,856 - [src.balancing.under_sampler] - INFO -   - Class 1.0: 13570 samples
2025-07-20 13:31:55,857 - [src.balancing.under_sampler] - INFO -   - Class 2.0: 90225 samples
2025-07-20 13:31:56,068 - [src.balancing.under_sampler] - INFO - New class distribution after Random Under Sampling:
2025-07-20 13:31:56,070 - [src.balancing.under_sampler] - INFO -   - Class 0.0: 300000 samples (-226321)
2025-07-20 13:31:56,071 - [src.balancing.under_sampler] - INFO -   - Class 1.0: 13570 samples (-0)
2025-07-20 13:31:56,072 - [src.balancing.under_sampler] - INFO -   - Class 2.0: 50000 samples (-40225)
2025-07-20 13:31:56,074 - [src.balancing.under_sampler] - INFO - Dataset size: 630116 -> 363

In [16]:
rus_y_train.value_counts()

Diabetes
0.0    300000
2.0     50000
1.0     13570
Name: count, dtype: int64

#### **4.2.2 TomekLinks**

In [17]:
tomek_X_train, tokek_y_train = under_sampling_balancer.apply_tomek_links(
    X_train,
    y_train,
    sampling_strategy=[0],
    n_jobs=os.cpu_count()
)

2025-07-20 13:31:56,106 - [src.balancing.under_sampler] - INFO - Starting Tomek Links process...
2025-07-20 13:31:56,123 - [src.balancing.under_sampler] - INFO - Original class distribution:
2025-07-20 13:31:56,125 - [src.balancing.under_sampler] - INFO -   - Class 0.0: 526321 samples
2025-07-20 13:31:56,126 - [src.balancing.under_sampler] - INFO -   - Class 1.0: 13570 samples
2025-07-20 13:31:56,133 - [src.balancing.under_sampler] - INFO -   - Class 2.0: 90225 samples
2025-07-20 13:44:10,422 - [src.balancing.under_sampler] - INFO - New class distribution after Tomek Links:
2025-07-20 13:44:10,422 - [src.balancing.under_sampler] - INFO -   - Class 0.0: 503935 samples (-22386 Tomek links)
2025-07-20 13:44:10,423 - [src.balancing.under_sampler] - INFO -   - Class 1.0: 13570 samples (unchanged)
2025-07-20 13:44:10,423 - [src.balancing.under_sampler] - INFO -   - Class 2.0: 90225 samples (unchanged)
2025-07-20 13:44:10,424 - [src.balancing.under_sampler] - INFO - Dataset size: 630116 -> 60

### **4.3 Hyprid**

In [18]:
hyprid_balancer = HybridSamplingBalancer()

2025-07-20 13:44:10,436 - [src.balancing.hyprid] - INFO - HybridSamplingBalancer initialized successfully


#### **4.3.1 SMOTE + TomekLinks**

In [20]:
smote_tomek_X_train, smote_tomek_y_train = hyprid_balancer.apply_smote_tomek(
    X_train,
    y_train,
    OVER_SAMPLING_STRATEGY,
    n_jobs=os.cpu_count()
)

2025-07-20 14:39:58,644 - [src.balancing.hyprid] - INFO - Starting SMOTETomek hybrid sampling process...
2025-07-20 14:39:58,653 - [src.balancing.hyprid] - INFO - Original class distribution:
2025-07-20 14:39:58,654 - [src.balancing.hyprid] - INFO -   - Class 0.0: 526321 samples
2025-07-20 14:39:58,655 - [src.balancing.hyprid] - INFO -   - Class 1.0: 13570 samples
2025-07-20 14:39:58,656 - [src.balancing.hyprid] - INFO -   - Class 2.0: 90225 samples
2025-07-20 14:58:42,967 - [src.balancing.hyprid] - INFO - New class distribution after SMOTETomek:
2025-07-20 14:58:42,969 - [src.balancing.hyprid] - INFO -   - Class 0.0: 509938 samples (-16383 net)
2025-07-20 14:58:42,970 - [src.balancing.hyprid] - INFO -   - Class 1.0: 49326 samples (+35756 net)
2025-07-20 14:58:42,970 - [src.balancing.hyprid] - INFO -   - Class 2.0: 84001 samples (-6224 net)
2025-07-20 14:58:42,971 - [src.balancing.hyprid] - INFO - Dataset size: 630116 -> 643265 samples (+13149)
2025-07-20 14:58:42,973 - [src.balancing.

In [21]:
smote_tomek_y_train.value_counts()

Diabetes
0.0    509938
2.0     84001
1.0     49326
Name: count, dtype: int64

#### **4.3.2 SMOTE + ENN**

In [22]:
smoteenn_X_train, smoteenn_y_train = hyprid_balancer.apply_smote_enn(
    X_train,
    y_train,
    OVER_SAMPLING_STRATEGY,
    os.cpu_count()
)

2025-07-20 14:58:43,022 - [src.balancing.hyprid] - INFO - Starting SMOTEENN hybrid sampling process...
2025-07-20 14:58:43,035 - [src.balancing.hyprid] - INFO - Original class distribution:
2025-07-20 14:58:43,037 - [src.balancing.hyprid] - INFO -   - Class 0.0: 526321 samples
2025-07-20 14:58:43,038 - [src.balancing.hyprid] - INFO -   - Class 1.0: 13570 samples
2025-07-20 14:58:43,040 - [src.balancing.hyprid] - INFO -   - Class 2.0: 90225 samples
2025-07-20 15:17:47,371 - [src.balancing.hyprid] - INFO - New class distribution after SMOTEENN:
2025-07-20 15:17:47,372 - [src.balancing.hyprid] - INFO -   - Class 0.0: 346080 samples (-180241 net)
2025-07-20 15:17:47,372 - [src.balancing.hyprid] - INFO -   - Class 1.0: 33113 samples (+19543 net)
2025-07-20 15:17:47,374 - [src.balancing.hyprid] - INFO -   - Class 2.0: 8812 samples (-81413 net)
2025-07-20 15:17:47,376 - [src.balancing.hyprid] - INFO - Dataset size: 630116 -> 388005 samples (-242111)
2025-07-20 15:17:47,377 - [src.balancing.hy

In [23]:
smoteenn_y_train.value_counts()

Diabetes
0.0    346080
1.0     33113
2.0      8812
Name: count, dtype: int64

## **3. Models**

### **3.1 SGD Classifier**

In [ ]:
sgd_model = DiabetesSGDClassifier(
    loss=SGD_LOSS,
    penalty=SGD_PENALTY,
    alpha=SGD_ALPHA,
    max_iter=SGD_MAX_ITER,
    random_state=RANDOM_STATE,
    learning_rate=SGD_LEARNING_RATE
)
sgd_model_info = sgd_model.get_model_info()

In [ ]:
sgd_model_info

In [ ]:
sgd_model.train(X_train, y_train)

In [ ]:
sgd_y_pred = sgd_model.predict(X_test)

In [ ]:
sgd_model.evaluate(y_test, sgd_y_pred)

In [ ]:
sgd_prob = sgd_model.predict_proba(X)

In [ ]:
plot_roc_auc(y, sgd_prob, "SGDClassifier")

### **3.2 Logistic Regression**

In [ ]:
lr_model = DiabetesLogisticRegression(C=LR_C, max_iter=LR_MAX_ITER, solver=LR_SOLVER)
lr_model_info = lr_model.get_model_info()

In [ ]:
lr_model.train(X_train, y_train)

In [ ]:
lr_y_pred = lr_model.predict(X_test)

In [ ]:
lr_model.evaluate(y_test, lr_y_pred)

In [ ]:
lr_prob = lr_model.predict_proba(X)

In [ ]:
plot_roc_auc(y, lr_prob, "LogisticRegression")

### **3.3 Naive Bayes**

In [ ]:
nb_model = DiabetesNaiveBayes(
    nb_type=NB_TYPE,
    var_smoothing=NB_VAR_SMOOTHING,
    binarize=NB_BINARIZE,
    alpha=NB_ALPHA
)
nb_model_info = nb_model.get_model_info()

In [ ]:
nb_model_info

In [ ]:
nb_model.train(X_train, y_train)

In [ ]:
nb_y_pred = nb_model.predict(X_test)

In [ ]:
nb_model.evaluate(y_test, nb_y_pred)

In [ ]:
nb_prob = nb_model.predict_proba(X)

In [ ]:
plot_roc_auc(y, nb_prob, "NaiveBayes")

### **3.4 Random Forest**

In [ ]:
rf_model = DiabetesRandomForest(
    n_estimators=RF_N_ESTIMATORS,
    max_depth=RF_MAX_DEPTH,
    random_state=RANDOM_STATE,
)
rf_model_info = rf_model.get_model_info()

In [ ]:
rf_model_info

In [ ]:
rf_model.train(X_train, y_train)

In [ ]:
rf_y_pred = rf_model.predict(X_test)

In [ ]:
rf_model.evaluate(y_test, rf_y_pred)

In [ ]:
rf_prob = rf_model.predict_proba(X)

In [ ]:
plot_roc_auc(y, rf_prob, "RandomForest")

## **4. Compare models**

In [ ]:
models = {
    "SGDClassifier": sgd_model,
    "LogisticRegression": lr_model,
    "NaiveBayes": nb_model,
    "RandomForest": rf_model
}

compare_results = compare_models(models, X_test, y_test)

In [ ]:
compare_results

In [ ]:
df["Diabetes"].value_counts()